# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Don't treat as dict/list
print(f"{getattr(metadata, 'name', '')}: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets by @id and name
recordset_ids = []

for recordset in getattr(metadata, 'recordSets', []):
    print(f"RecordSet name: {getattr(recordset, 'name', '')}")
    print(f"  @id: {getattr(recordset, '@id', '')}")
    print("  Fields:")
    for field in getattr(recordset, 'fields', []):
        print(f"    - {getattr(field, 'name', '')} (@id: {getattr(field, '@id', '')})")
    recordset_ids.append(getattr(recordset, '@id', ''))
    print()

# If no recordSets are found, print a message
if not recordset_ids:
    print("No record sets found. Please check if the dataset provides recordSet definitions.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For demonstration, pick the first record set if available
if recordset_ids:
    main_recordset_id = recordset_ids[0]
    print(f"Loading data for record set {main_recordset_id} ...")
    # The mlcroissant API expects record_set to be the @id
    records = list(dataset.records(record_set=main_recordset_id))
    df = pd.DataFrame(records)
    print("Available columns (fields by @id):", df.columns.tolist())
    df.head()
else:
    df = pd.DataFrame()
    print("No data to load; skipping data extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

if not df.empty:
    # Heuristics: find a likely numeric field by looking for fields containing 'int', 'float', or known numeric field names
    numeric_fields = [col for col in df.columns if df[col].dtype in [np.int64, np.float64, float, int]]
    # If not autodetect, fall back to a common mass spec field, for example 'cr:Coverage' or 'cr:MolecularWeight', etc.
    fallbacks = [col for col in df.columns if any(x in col.lower() for x in ['coverage', 'abundance', 'count', 'mw', 'weight', 'cr:coverage', 'cr:molecularweight', 'cr:peptidecount'])]
    numeric_field = numeric_fields[0] if numeric_fields else (fallbacks[0] if fallbacks else df.columns[0])
    print(f"Selected numeric field: {numeric_field}")
    
    # Pick a threshold based on the field's quantile
    threshold = df[numeric_field].quantile(0.75) if np.issubdtype(df[numeric_field].dtype, np.number) else None
    # Use a simple filter (if possible)
    if threshold is not None:
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    else:
        print(f"No numeric field detected for analysis.")

    # Try grouping by a categorical field, e.g., sample, or other text fields, select a likely one
    group_fields = [col for col in df.columns if df[col].nunique() < 20 and df[col].dtype == object and col != numeric_field]
    group_field = group_fields[0] if group_fields else None
    if group_field is not None and threshold is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped data by {group_field}:\n", grouped_df.head())
    else:
        print("No suitable group field found for grouping.")
else:
    print("No data for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

if not df.empty and 'numeric_field' in locals():
    # Histogram
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field], kde=True)
    plt.title(f'Histogram of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by group if possible
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f'{numeric_field} grouped by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
    
else:
    print("No data to visualize.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded and explored the FAIR² mass spectrometry dataset of human mast cell extracellular vesicles using the `mlcroissant` library. 

We reviewed the schema, loaded record sets by their `@id`, and demonstrated filtering, normalization, and grouping by field. We also visualized data distributions for quantitative fields. This approach can be extended to further analyze the protein-level data, study abundances, and discover biological insights present in the dataset.

For reproducible workflows, always reference fields and record sets by their unique `@id` as shown.